In [1]:
import torch
import torchvision.models as models
import torch.nn as nn
from google.colab import drive
import os
from PIL import Image
import torchvision.transforms as transforms

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Re-initialize the Model Architecture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(weights=None)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 200)

# 3. Load Your Fine-Tuned Weights from Day 1
model_path = '/content/drive/MyDrive/XAI_Project/resnet50_cub200_finetuned.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
model.eval() # CRITICAL: Put model in evaluation mode for XAI
print("Fine-tuned model successfully loaded and ready for XAI!")

# 4. Standard Eval Transforms (NO Data Augmentation here!)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Mounted at /content/drive
Fine-tuned model successfully loaded and ready for XAI!


In [2]:
import numpy as np
import os
import pandas as pd
from tqdm import tqdm

# 1. Setup paths
csv_path = '/content/drive/MyDrive/XAI_Project/confident_subset.csv'
confident_df = pd.read_csv(csv_path)

lime_dir = '/content/drive/MyDrive/XAI_Project/LIME_Heatmaps'
lrp_dir = '/content/drive/MyDrive/XAI_Project/LRP_Heatmaps'
consensus_dir = '/content/drive/MyDrive/XAI_Project/Consensus_Maps'
os.makedirs(consensus_dir, exist_ok=True)

# 2. Mathematical Normalization Function
def min_max_normalize(heatmap):
    """Scales all values in a heatmap to perfectly fit between 0.0 and 1.0"""
    h_min, h_max = np.min(heatmap), np.max(heatmap)
    # Add a tiny epsilon (1e-8) to prevent dividing by zero if a map is completely blank
    return (heatmap - h_min) / (h_max - h_min + 1e-8)

print("Calculating Mathematical Consensus Maps...")

# 3. The Aggregation Loop
for index, row in tqdm(confident_df.iterrows(), total=len(confident_df)):
    dataset_idx = int(row['dataset_idx'])

    # Load the raw arrays you generated on Day 2
    lime_path = os.path.join(lime_dir, f'lime_mask_{dataset_idx}.npy')
    lrp_path = os.path.join(lrp_dir, f'lrp_heatmap_{dataset_idx}.npy')

    # We use a try-except block just in case a file failed to save properly
    try:
        lime_raw = np.load(lime_path)
        lrp_raw = np.load(lrp_path)

        # Normalize both maps to the same 0-1 scale
        lime_norm = min_max_normalize(lime_raw)
        lrp_norm = min_max_normalize(lrp_raw)

        # Element-wise multiplication to isolate features where BOTH methods agree
        consensus_map = lime_norm * lrp_norm

        # Save the final consensus map!
        save_path = os.path.join(consensus_dir, f'consensus_map_{dataset_idx}.npy')
        np.save(save_path, consensus_map)

    except FileNotFoundError:
        print(f"Skipping index {dataset_idx}: Missing a LIME or LRP file.")

print("\nAll Consensus Maps generated!")

Calculating Mathematical Consensus Maps...


100%|██████████| 600/600 [04:45<00:00,  2.10it/s]


All Consensus Maps generated!
